# STIRPAT과 로그회귀

## 국가·기간 표본을 바꾼 탄력성 추정

강의에서 확인한 기본 회귀와 패널자료의 순서를 유지하면서, 선택한 국가·기간에서 통합 회귀 기준선과 국가·연도 고정효과 모형을 비교합니다.


## 이번 실습에서 완성할 것

- 통합 회귀와 고정효과가 쓰는 비교정보를 구분합니다.
- 계수와 표준오차를 따로 읽습니다.
- 표본 변경 전후의 계수 민감성을 확인합니다.

**완료 결과:** 두 명세 계수표, 표본 민감성 결과, 조건부 연관성으로 작성한 문장


## 이번 교시에서 사용할 데이터

### STIRPAT 수업용 국가 패널

| 항목 | 내용 |
|---|---|
| 구성 | 한국을 포함한 12개국 |
| 기간과 규모 | 1990~2022년, 396행의 균형 패널 |
| 한 행의 의미 | 한 국가의 한 연도 환경·인구·경제 관측 |
| 결과변수 | `co2_tonnes` |
| 설명변수 | `population`, `gdp_per_capita_constant_2021_intl` |
| provenance 기록 | `gdp_value_status` |

이 자료는 공개 지표를 수업 명세에 맞춰 정리한 snapshot이며 York 등의 원 논문 표본을 복제한 자료가 아닙니다. 금액 변수는 constant 2021 international dollars 기준이므로 명목 GDP와 혼용하지 않습니다.

이 자료에는 같은 국가가 여러 연도에 반복됩니다. 따라서 국가가 다른 행과 같은 국가의 연도가 다른 행을 구분해서 읽어야 합니다. 강의에서는 12개국 전체를 사용하고, 적용 실습에서는 국가 목록 또는 기간 하나만 변경합니다. 회귀명세와 표준오차 조정은 자료의 행 구조를 확인한 뒤 단계적으로 소개합니다.


## 이번 실습의 분석 순서

먼저 제공된 네 국가·기간에서 pooled OLS를 기준선으로 적합하고, 국가·연도 고정효과 결과와 비교합니다. 그다음 표본기간 또는 국가 목록 가운데 하나만 바꾸어 계수의 민감성을 확인합니다.

**남길 결과:** 두 명세의 점추정 계수표, 표본변경 민감성 표, 탄력성을 조건부 연관성으로 서술한 문장


## 실습 구간 안내

| 구분 | 수행 범위 |
|---|---|
| **수업 중 필수** | pooled OLS와 고정효과 점추정 비교·PASS |
| **시간이 남으면** | 표본기간을 바꾼 pooled OLS 민감성 |
| **수업 후 확장(선택)** | 국가 하나씩 제외 및 표준오차 계산법 비교 |

수업 중에는 필수 구간의 결과와 마지막 `PASS`를 먼저 완성합니다. 확장 구간을 실행하지 않아도 필수 셀이 독립적으로 작동하도록 구성되어 있습니다.


## 실행 환경


In [ ]:
from pathlib import Path
import platform
import warnings

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 130)
warnings.filterwarnings("ignore", category=FutureWarning)

def find_course_data(relative_path: str) -> Path:
    """0교시에서 준비한 data 폴더의 파일을 찾습니다."""
    relative = Path(relative_path)
    checked = []
    for start in [Path.cwd(), *Path.cwd().parents, Path("/content")]:
        candidate = (start / "data" / relative).resolve()
        if candidate in checked:
            continue
        checked.append(candidate)
        if candidate.exists():
            return candidate
    raise FileNotFoundError(
        f"데이터 파일을 찾지 못했습니다: {relative_path}. "
        "과정 폴더의 00_setup.ipynb를 먼저 실행해 주세요."
    )
print("Python:", platform.python_version())
print("pandas:", pd.__version__)


## 1. 패널자료의 로그변수 생성

국가 × 연도 패널에서 CO₂, 인구, 1인당 GDP를 선택하고 양수 여부를 확인한 뒤 자연로그를 만듭니다.


In [ ]:
import statsmodels.formula.api as smf

PANEL_PATH = find_course_data("methods/stirpat_panel.csv")
panel = pd.read_csv(PANEL_PATH)
SELECTED_CODES = ['KOR', 'JPN', 'CHN', 'IND']
START_YEAR = 2000
END_YEAR = 2022
analysis_name = "STIRPAT East Asia sensitivity analysis"

analysis = (
    panel.loc[panel["code"].isin(SELECTED_CODES) & panel["year"].between(START_YEAR, END_YEAR)]
         .copy()
         .assign(
             ln_co2=lambda d: np.log(d["co2_tonnes"]),
             ln_population=lambda d: np.log(d["population"]),
             ln_affluence=lambda d: np.log(d["gdp_per_capita_constant_2021_intl"]),
             year_centered=lambda d: d["year"] - d["year"].min(),
         )
)
print("shape:", analysis.shape)
print("countries:", sorted(analysis["code"].unique()))
display(analysis.head(3))
print("\n[같은 관측의 원래 값과 로그 값]")
display(analysis[["code", "year", "co2_tonnes", "ln_co2", "population", "ln_population",
                  "gdp_per_capita_constant_2021_intl", "ln_affluence"]].head(6))
print("로그를 취해도 한 행은 여전히 한 국가의 한 연도입니다.")


## 2. 회귀명세의 단계별 비교

강의와 같은 순서를 유지합니다. 먼저 선택한 국가–연도 행 전체에 공통 회귀식 하나를 적합해 실제값·적합값·잔차를 확인합니다. 그다음 국가·연도 고정효과를 추가해 점추정 계수가 어떻게 달라지는지 비교합니다. 표준오차 조정은 필수 분석 뒤의 선택 심화에서 확인하며, 국가 수가 너무 적으면 cluster 표준오차를 보고하지 않습니다.

> **실행 전 확인:** 이번에 바꾼 국가 또는 기간 조건 하나를 기록하고, pooled와 고정효과가 같은 원자료 행을 사용하는지 확인해 주세요.


### 2-1. Pooled OLS 기준선과 잔차

선택한 국가–연도 행을 하나의 표본으로 결합해 공통 회귀식을 적합합니다. 실제 로그 CO₂와 적합값의 차이를 잔차로 정의하고, OLS가 최소화하는 잔차제곱합을 확인합니다.

$$e_i=y_i-\hat y_i, \qquad SSE=\sum_i e_i^2$$

이 단계에서는 고정효과와 표준오차 조정을 적용하지 않습니다. Pooled OLS는 이후 명세와 비교할 기준모형입니다.


In [ ]:
pooled_formula = "ln_co2 ~ ln_population + ln_affluence"
pooled_model = smf.ols(pooled_formula, data=analysis)
pooled_plain = pooled_model.fit()

pooled_preview = analysis[["code", "year", "ln_co2"]].copy()
pooled_preview["fitted"] = pooled_plain.fittedvalues
pooled_preview["residual"] = pooled_preview["ln_co2"] - pooled_preview["fitted"]
pooled_sse = float(np.square(pooled_preview["residual"]).sum())

print("[pooled OLS에 넣은 회귀식]")
print(pooled_formula)
print(f"관측 행 수: {int(pooled_plain.nobs)}")
print("\n[실제값 - 적합값 = 잔차]")
display(pooled_preview.head(8).round(4))
print(f"잔차 제곱합 SSE: {pooled_sse:.4f}")

pooled_point_table = pd.DataFrame({
    "term": ["ln_population", "ln_affluence"],
    "coefficient": [pooled_plain.params["ln_population"], pooled_plain.params["ln_affluence"]],
})
print("\n[pooled OLS 점추정 계수]")
display(pooled_point_table.round(4))


## 시간이 남으면 — 표본기간을 바꾼 Pooled OLS 계수 비교

국가 목록은 유지하고 기간만 바꿉니다. 인구·소득 탄력성 계수가 기간에 따라 얼마나 달라지는지 확인하고, 이를 ‘표본기간 민감성’으로 기록하세요.


In [ ]:
period_rows = []
for period_start, period_end in [(1990, 2005), (2006, 2022), (2000, 2022)]:
    period_data = (
        panel.loc[
            panel["code"].isin(SELECTED_CODES)
            & panel["year"].between(period_start, period_end)
        ]
        .copy()
        .assign(
            ln_co2=lambda d: np.log(d["co2_tonnes"]),
            ln_population=lambda d: np.log(d["population"]),
            ln_affluence=lambda d: np.log(d["gdp_per_capita_constant_2021_intl"]),
        )
    )
    period_fit = smf.ols("ln_co2 ~ ln_population + ln_affluence", data=period_data).fit()
    period_rows.append({
        "period": f"{period_start}–{period_end}",
        "rows": int(period_fit.nobs),
        "countries": period_data["code"].nunique(),
        "population_elasticity": period_fit.params["ln_population"],
        "affluence_elasticity": period_fit.params["ln_affluence"],
        "R2": period_fit.rsquared,
    })

period_sensitivity = pd.DataFrame(period_rows)
display(period_sensitivity.round(3))
print("계수 차이는 선택한 국가와 기간의 조건부 연관성이 달라졌음을 보여 줍니다.")


## 수업 후 확장(선택) — 국가를 하나씩 제외한 계수 민감성

필수 실습을 마친 뒤 수행합니다. 전체 선택 국가로 적합한 계수와, 국가를 하나씩 제외해 다시 적합한 계수를 비교합니다. 특정 국가를 제외했을 때 계수가 크게 달라진다면 축소 표본의 구성에 민감한 결과로 기록합니다.


In [ ]:
loo_codes = list(SELECTED_CODES)
assert len(loo_codes) >= 4, "leave-one-country-out 비교에는 네 국가 이상이 필요합니다."

loo_data = (
    panel.loc[
        panel["code"].isin(loo_codes)
        & panel["year"].between(START_YEAR, END_YEAR)
    ]
    .copy()
    .assign(
        ln_co2=lambda d: np.log(d["co2_tonnes"]),
        ln_population=lambda d: np.log(d["population"]),
        ln_affluence=lambda d: np.log(d["gdp_per_capita_constant_2021_intl"]),
    )
)

loo_rows = []
for omitted_code in [None, *loo_codes]:
    fit_data = loo_data if omitted_code is None else loo_data.loc[loo_data["code"].ne(omitted_code)]
    fit = smf.ols("ln_co2 ~ ln_population + ln_affluence", data=fit_data).fit()
    loo_rows.append({
        "sample": "all selected countries" if omitted_code is None else f"omit {omitted_code}",
        "countries": fit_data["code"].nunique(),
        "rows": int(fit.nobs),
        "population_elasticity": fit.params["ln_population"],
        "affluence_elasticity": fit.params["ln_affluence"],
        "R2": fit.rsquared,
    })

loo_sensitivity = pd.DataFrame(loo_rows).set_index("sample")
full_population = loo_sensitivity.loc["all selected countries", "population_elasticity"]
full_affluence = loo_sensitivity.loc["all selected countries", "affluence_elasticity"]
loo_sensitivity["population_change_from_full"] = loo_sensitivity["population_elasticity"] - full_population
loo_sensitivity["affluence_change_from_full"] = loo_sensitivity["affluence_elasticity"] - full_affluence

print("[국가 하나씩 제외한 pooled OLS 계수]")
display(loo_sensitivity.round(3))
largest_shift = (
    loo_sensitivity.drop(index="all selected countries")[[
        "population_change_from_full", "affluence_change_from_full"
    ]]
    .abs()
    .max(axis=1)
    .idxmax()
)
print("전체 표본 대비 계수 변화가 가장 큰 제외 조건:", largest_shift)
print("이 표는 축소 표본 구성에 대한 점추정의 민감성을 보여 주며 인과효과나 유의성을 판정하지 않습니다.")


### 2-2. 국가·연도 고정효과와 비교정보

이번에는 설명변수는 그대로 두고 `C(code)`와 `C(year)`를 추가합니다.

- `C(code)`: 국가마다 다른 기본 높이를 허용해 시간에 거의 변하지 않는 국가 차이를 분리합니다.
- `C(year)`: 같은 해 여러 국가가 함께 경험한 공통 충격을 분리합니다.

두 명세의 점추정 계수 차이는 계수를 구성하는 비교정보가 달라졌음을 보여줍니다. 계수 차이만으로 어느 명세가 인과적으로 타당한지 판단할 수는 없습니다.


In [ ]:
fixed_effects_formula = "ln_co2 ~ ln_population + ln_affluence + C(code) + C(year)"
fixed_effects_model = smf.ols(fixed_effects_formula, data=analysis)
fixed_effects_plain = fixed_effects_model.fit()

terms = ["ln_population", "ln_affluence"]
point_estimate_table = pd.DataFrame({
    "term": terms,
    "pooled_coef": [pooled_plain.params[term] for term in terms],
    "fixed_effects_coef": [fixed_effects_plain.params[term] for term in terms],
})
point_estimate_table["difference_FE_minus_pooled"] = (
    point_estimate_table["fixed_effects_coef"] - point_estimate_table["pooled_coef"]
)

print("[두 모형의 점추정 계수 비교]")
display(point_estimate_table.round(4))
print("pooled는 국가 간 차이와 국가 내 변화를 함께 사용합니다.")
print("국가·연도 고정효과는 같은 국가 안의 변화가 더 직접적으로 반영되게 합니다.")
print("계수 차이만으로 어느 모형이 인과적으로 옳다고 판정하지 않습니다.")


### 선택 심화 — 2-3. 표준오차 계산법에 따른 불확실성

`HC3`와 국가 단위 `cluster`는 같은 회귀식의 **점추정 계수**를 바꾸는 방법이 아니라 **표준오차 계산 방식**을 조정하는 방법입니다.

강의 표본은 국가가 12개뿐이므로 cluster 표준오차를 정밀한 결론처럼 읽지 않습니다. 독립 실습처럼 국가가 10개 미만이면 이 Notebook은 cluster 표준오차를 계산 결과표에 제시하지 않고 `NaN`으로 남깁니다.


In [ ]:
pooled_hc3 = pooled_model.fit(cov_type="HC3")
cluster_count = int(analysis["code"].nunique())

if cluster_count >= 10:
    fixed_effects_cluster = fixed_effects_model.fit(
        cov_type="cluster",
        cov_kwds={"groups": analysis["code"]},
    )
    fixed_effects = fixed_effects_cluster
    fixed_effects_se_values = [fixed_effects_cluster.bse[term] for term in terms]
    cluster_status = f"계산함 — 국가 cluster {cluster_count}개이므로 결과는 주의해서 해석"
else:
    fixed_effects_cluster = None
    fixed_effects = fixed_effects_plain
    fixed_effects_se_values = [np.nan for _ in terms]
    cluster_status = f"보고하지 않음 — 국가 cluster가 {cluster_count}개뿐임"

pooled = pooled_hc3
result_table = pd.DataFrame({
    "term": terms,
    "pooled_coef": [pooled_plain.params[term] for term in terms],
    "pooled_se_HC3": [pooled_hc3.bse[term] for term in terms],
    "fixed_effects_coef": [fixed_effects_plain.params[term] for term in terms],
    "fixed_effects_se_cluster": fixed_effects_se_values,
})
# 뒤의 진단·검토 셀이 사용하는 짧은 열 이름도 유지합니다.
result_table["pooled_se"] = result_table["pooled_se_HC3"]
result_table["fixed_effects_se"] = result_table["fixed_effects_se_cluster"]

uncertainty_table = pd.DataFrame({
    "term": terms,
    "pooled_default_se": [pooled_plain.bse[term] for term in terms],
    "pooled_HC3_se": [pooled_hc3.bse[term] for term in terms],
    "FE_default_se": [fixed_effects_plain.bse[term] for term in terms],
    "FE_country_cluster_se": fixed_effects_se_values,
})

print("[점추정 계수와 조정 표준오차]")
display(result_table[["term", "pooled_coef", "pooled_se_HC3", "fixed_effects_coef", "fixed_effects_se_cluster"]].round(4))
print("\n[표준오차 계산방식 비교]")
display(uncertainty_table.round(4))
print("cluster 표준오차:", cluster_status)
print("표준오차 조정 전후의 점추정 계수는 같습니다:",
      np.allclose(pooled_plain.params[terms], pooled_hc3.params[terms]))


## 3. 표본 구조와 잔차 진단

계수표만 보지 않고 fitted–residual 패턴, 국가별 관측 수, 설명변수 상관을 확인합니다.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

diagnostics = analysis[["code", "year", "ln_population", "ln_affluence"]].copy()
diagnostics["fitted"] = fixed_effects_plain.fittedvalues
diagnostics["residual"] = fixed_effects_plain.resid

coverage = analysis.groupby("code").agg(first_year=("year", "min"), last_year=("year", "max"), rows=("year", "size"))
display(coverage)
print("correlation ln_population vs ln_affluence:", analysis[["ln_population", "ln_affluence"]].corr().iloc[0, 1])

sns.set_theme(style="whitegrid", context="notebook")
fig, ax = plt.subplots(figsize=(7.5, 4.5))
sns.scatterplot(data=diagnostics, x="fitted", y="residual", hue="code", alpha=0.75, ax=ax)
ax.axhline(0, color="black", linewidth=0.8)
ax.set(title="Fixed-effects residual diagnostic", xlabel="Fitted log CO2", ylabel="Residual")
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()


## 4. 모형 비교 조건 검증

모든 국가가 같은 기간을 갖는지, 핵심 계수가 추정됐는지, 잔차가 유한한지 확인합니다.


In [ ]:
checks = pd.Series({
    "selected countries present": set(analysis["code"]) == set(SELECTED_CODES),
    "balanced selected panel": coverage["rows"].nunique() == 1,
    "positive original variables": (analysis[["co2_tonnes", "population", "gdp_per_capita_constant_2021_intl"]] > 0).all().all(),
    "pooled residual identity": np.allclose(
        pooled_preview["ln_co2"] - pooled_preview["fitted"], pooled_preview["residual"]
    ),
    "core coefficients estimated": point_estimate_table[["pooled_coef", "fixed_effects_coef"]].notna().all().all(),
    "finite residuals": np.isfinite(diagnostics["residual"]).all(),
}, name="passed")
display(checks.to_frame())
assert checks.all()
print("회귀 비교조건 검증: PASS")


## 결과 기록과 ChatGPT 검토

다음 셀은 실제 표본·기간, pooled OLS와 고정효과 점추정 계수표, 표본범위와 검증표를 직접 넣습니다.

- **관찰:** 두 명세의 탄력성 계수와 차이
- **조건:** 국가–연도 표본, P–A 축소 명세, 로그 단위
- **한계:** 누락요인·역인과와 현재 표에 없는 유의성 정보


In [ ]:
review_prompt = f"""
STIRPAT P–A 축소 명세 결과를 다음 실제 출력만 사용해 검토해 주세요.

- 국가: {sorted(analysis['code'].unique())}
- 기간: {analysis['year'].min()}~{analysis['year'].max()}
- 관측 행 수: {len(analysis)}
- pooled OLS와 고정효과 점추정 계수:
{point_estimate_table.to_string(index=False)}
- 국가별 표본범위:
{coverage.to_string()}
- 필수 검사:
{checks.to_string()}

관찰 | 분석조건 | 해석한계의 세 항목으로 답해 주세요. 계수는 1% 변화 단위의
조건부 탄력성으로 읽고, 제공하지 않은 표준오차·p-value·유의성을 만들지
마세요. 고정효과만으로 인과효과가 증명됐다고 쓰지 마세요.
""".strip()
print("[ChatGPT에 복사할 교시별 검토 프롬프트]\n")
print(review_prompt)


## 완료 확인

마지막 출력이 `회귀 비교조건 검증: PASS`인지 확인해 주세요. 이 검사는 필요한 계산이 실행됐는지 확인할 뿐, 연구 주장의 타당성을 자동 판정하지 않습니다.


## 결과를 정리할 질문

- 어떤 국가 또는 기간을 바꾸었으며 pooled·고정효과 계수는 어떻게 달라졌습니까?
- 두 명세가 사용하는 비교정보의 차이를 결과문장에 반영했습니까?
- 계수를 탄력성의 조건부 연관성으로 표현하고 인과효과로 단정하지 않았습니까?
